<a href="https://colab.research.google.com/github/GitTanish/RAG/blob/master/RagAPP27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import langchain
print(langchain.__version__)

In [ ]:
import os
os.environ['GROQ_API_KEY']=""

In [ ]:
os.environ["LANGCHAIN_TRACING"]="true"
os.environ["LANGCHAIN_API_KEY"]=""
os.environ["LANGCHAIN_PROJECT"]="langchain101"

In [ ]:
! pip install -qU langchain-groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")
llm_response = llm.invoke('Tell me a joke')
llm_response

#### Parsing output


In [ ]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
output_parser.invoke(llm_response)

#### Simple Chain

In [ ]:
chain = llm | output_parser
chain.invoke("How make a scrambled egg")

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class WorkOutPlanner(BaseModel):
  routine: str = Field(description = "When does the speaker wakes and sleep")
  shoulder_exercise: str = Field(description="What are his shoulder specific exercise")
  diet: str = Field(description= "What food he eats for his diet")

transcription = """
Hey It's me XYZ, My day starts at waking up on 7 am , I start my day with scrambled eggs, milk and bread.
Then I work on my various project, at 3 pm. I hit the gym for workout. For chest I do bench press, I can bench a small house I feel.
For shoulders I do Shoulder press and Lateral Raises feels great tho. Like wise I divide the week different muscle groups.

I normally eat chicken breast with some butter , Mackerel and sometimes lobster.
I feel alive and great.

"""

structured_llm = llm.with_structured_output(WorkOutPlanner)
output = structured_llm.invoke(transcription)
output

In [ ]:
output.diet[0:14]

#### Prompt Template

In [ ]:
# dynamic prompting
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Best thing about {topic}, in 100 words")
prompt.invoke({"topic":"programming"})

In [ ]:
chain = prompt | llm | output_parser
chain.invoke({"topic":"Flowerhorn"})

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# define the prompt
prompt = ChatPromptTemplate.from_template("Technical niche about {topic}")

# initialise the llm
llm = ChatGroq(model = "openai/gpt-oss-120b")

# Define the output parser
output_parser = StrOutputParser()

# compose the chain
chain = prompt | llm | output_parser

# use the chain
result = chain.invoke({"topic":"Cockatoo"})
print(result)

#### LLM Messages

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

system_message = SystemMessage(content= "You are an edgy internet tech influencer that does improv, you're job here is to make joke of a query")
human_message = HumanMessage(content = 'Tell me about programming')
llm.invoke([system_message, human_message])

In [ ]:
template = ChatPromptTemplate([
    ('system', "You're a struggling comedian that tells jokes."),
    ("human","tell me about {topic}")
])

prompt_value = template.invoke(
    {
        "topic":"Fish Curry"
    }
)

llm.invoke(prompt_value)

In [ ]:
prompt_value

In [ ]:
! pip install docx2txt pypdf unstructured

In [ ]:
!pip install -U langchain langchain-community sentence-transformers

In [ ]:
!pip install docx2txt

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import List
from langchain_core.documents import Document

# splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    length_function=len
)

# load docx
docx_loader = Docx2txtLoader("/content/docs/rag_test_doc_1.docx")
documents = docx_loader.load()

# split
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks")



In [ ]:
splits[0]

In [ ]:
# function to load documents from a folder
def load_documents(folder_path: str) -> List[Document]:
  documents = []
  for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if filename.endswith('.pdf'):
      loader = PyPDFLoader(file_path)
    elif filename.endswith('.docx'):
      loader = Docx2txtLoader(file_path)
    else:
      print(f"Unsupported file type: {filename}")
      continue
    documents.extend(loader.load())
  return documents

# load documents from a folder
folder_path = "/content/docs"
documents = load_documents(folder_path)

print(f"loaded {len(documents)} documents from the folder")
splits = text_splitter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks")


In [ ]:
# 38:01

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# init model
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/e5-base-v2"
)

document_embeddings = embeddings.embed_documents(
    [f"passage: {split.page_content}" for split in splits]
)


print(f"Created embeddings for {len(document_embeddings)} documents chunks.")

In [ ]:
document_embeddings[0]

#### Create and persist Chroma vector store

In [ ]:
pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

embedding_function = HuggingFaceEmbeddings(
    model_name="intfloat/e5-base-v2"
)

collection_name = "my_collection"
vectorstore = Chroma.from_documents(collection_name=collection_name, documents= splits,
                                    embedding=embedding_function, persist_directory="./chroma_db")

print("Vector store created and persisted to './chroma_db'")

In [ ]:
# perform the similarity search

query = "Applications of AI in healthcare?"
search_results = vectorstore.similarity_search(query, k=2)

print(f"\nTop 2 most relevant chunks for the query: {query}\n")
for i, result in enumerate(search_results, 1):
  print(f"Result {i}:")
  print(f"Source: {result.metadata.get('source', 'unknown')}")
  print(f"Content: {result.page_content}")
  print()

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})
retriever.invoke("What can machine learning models analyze?")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
template= """Answer the question based only on the following context:
{context}

Question: {question}

Answer: """
prompt = ChatPromptTemplate.from_template(template)


In [ ]:
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context":retriever, "question":RunnablePassthrough()} | prompt
)
rag_chain.invoke("What can machine learning models analyze?")

In [ ]:
def docs2str(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
rag_chain = (
    {"context":retriever | docs2str, "question":RunnablePassthrough()} | prompt
)
rag_chain.invoke("What can machine learning models analyze?")

In [ ]:
rag_chain = (
    {"context":retriever| docs2str, "question":RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

question = "Features of Blockchain?"
response = rag_chain.invoke(question)
print(response)

#### Conversational RAG

- Handling follow Up Questions

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
chat_history = []
chat_history.extend([
    HumanMessage(content= question),
    AIMessage(content = response)
])


In [ ]:
chat_history

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question"
    "which might reference context in the chat history,"
    "formulate a standalone question which can be understood"
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as it."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder('chat_history'),
        ("human","{input}"),

    ]
)

contextualize_chain = contextualize_q_prompt | llm | StrOutputParser()
contextualize_chain.invoke({"input": "ok list the features of blockchain", "chat_history": chat_history})

In [ ]:
!pip install langchain-classic


In [ ]:
from langchain_classic.chains import create_history_aware_retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

history_aware_retriever.invoke({
    "input": "ok list the features of blockchain.",
    "chat_history": chat_history
})

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_messages([
    ("system","You are an helpful AI assistant. Use the following context to answer the user's question."),
    ("system","Context: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

question_answer_chain= create_stuff_documents_chain(llm, qa_prompt)

rag_chain= create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [ ]:
rag_chain.invoke({"input":"State only one feature of blockchain", "chat_history":chat_history})

#### Building Multi User Chatbot

In [ ]:
import sqlite3

DB_NAME = "rag_app.db"

def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    return conn

def create_application_logs():
    conn = get_db_connection()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS application_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT,
            user_query TEXT,
            gpt_response TEXT,
            model TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.close()

def insert_application_logs(session_id, user_query, gpt_response, model):
    conn = get_db_connection()
    conn.execute(
        'INSERT INTO application_logs (session_id, user_query, gpt_response, model) VALUES (?,?,?,?)',
        (session_id, user_query, gpt_response, model)
    )
    conn.commit()
    conn.close()

def get_chat_history(session_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        'SELECT user_query, gpt_response FROM application_logs WHERE session_id=? ORDER BY created_at',
        (session_id,)
    )

    messages = []
    for row in cursor.fetchall():
        messages.extend([
            {"role": "human", "content": row['user_query']},
            {"role": "ai", "content": row['gpt_response']}
        ])

    conn.close()
    return messages

create_application_logs

In [ ]:
import sqlite3
import uuid

create_application_logs()

session_id = str(uuid.uuid4())

# first query
chat_history = get_chat_history(session_id)

question1 = "What are the challenges mentioned?"
answer1 = rag_chain.invoke({
    "input": question1,
    "chat_history": chat_history
})['answer']

insert_application_logs(session_id, question1, answer1, "gpt-oss-120b")

print(answer1)

# second query (memory test)
chat_history = get_chat_history(session_id)

question2 = "summarize them"
answer2 = rag_chain.invoke({
    "input": question2,
    "chat_history": chat_history
})['answer']

insert_application_logs(session_id, question2, answer2, "gpt-oss-120b")

print(answer2)

In [ ]:
!pip install nbformat